In [1]:
import os
from dotenv import load_dotenv
from gitsource import GithubRepositoryDataReader

# Load environment variables
load_dotenv()



True

In [2]:
# Initialize the GitHub reader for the lesson markdown files
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()


# Q1. How many lesson pages


In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)
print(f"Documents loaded: {len(documents)}")    


Documents loaded: 72


# Q2: Indexing and Searching with minsearch

### How does the agentic loop keep calling the model until it stops?

### What's the filename of the first result?

In [4]:
import minsearch

index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

query = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    query=query,
    num_results=3
)


if search_results:
    print(f"Top Search Result Filename:\n{search_results[0]['filename']}")
else:
    print("No search results found.")

Top Search Result Filename:
01-agentic-rag/lessons/14-agentic-loop.md


# Q3: RAG with Token Counting

### we need to build a RAG pipeline and measure how many input (prompt) tokens are sent to the model using google-genai client.

In [5]:
from google import genai
from google.genai import types





In [6]:

client = genai.Client()

def build_prompt(query, search_results):
    """Builds the context prompt matching standard RAG patterns"""
    context = ""
    for doc in search_results:
        context += f"Filename: {doc['filename']}\nContent: {doc['content']}\n\n"
        
    prompt = f"""
You are a course teaching assistant. Answer the student's question based strictly on the context provided.

Context:
{context}

Question: {query}
Answer:
""".strip()
    return prompt

In [7]:
query = "How does the agentic loop keep calling the model until it stops?"
search_results = index.search(query=query, num_results=3)

prompt_text = build_prompt(query, search_results)

response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=prompt_text
)

input_tokens = response.usage_metadata.prompt_token_count
print(f"--- RAG Response ---")
print(response.text[:200] + "...\n")
print(f"--- Token Metrics ---")
print(f"Input (Prompt) Tokens Sent: {input_tokens}")

--- RAG Response ---
The agentic loop keeps calling the model until it stops by wrapping the process in a `while` loop.

Here's how it works:
1.  A `has_function_calls` flag is used to track if the model's response includ...

--- Token Metrics ---
Input (Prompt) Tokens Sent: 6417


# Q4: Chunking the Documents


In [8]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

print(f"chunks generated: {len(chunks)}")

chunks generated: 295


# Q5. RAG with chunking

In [9]:
# 1. Initialize a brand new minsearch index for the chunks
chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

# 2. Fit it with the 295 chunks
chunk_index.fit(chunks)

# 3. Run the exact same query
query = "How does the agentic loop keep calling the model until it stops?"
chunk_search_results = chunk_index.search(query=query, num_results=3)

# 4. Build the prompt using the chunk context
chunk_prompt_text = build_prompt(query, chunk_search_results)

# 5. Send to Gemini
chunk_response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=chunk_prompt_text
)

# 6. Extract the new token count
chunk_input_tokens = chunk_response.usage_metadata.prompt_token_count
print(f"Original Full Page Input Tokens (Q3): {input_tokens}")
print(f"New Chunked Input Tokens (Q5): {chunk_input_tokens}")

Original Full Page Input Tokens (Q3): 6417
New Chunked Input Tokens (Q5): 1665


#### 3× fewer token 

# Q6. Turning it into an agent


In [10]:
search_call_count = 0

def search(query: str) -> list:
    """
    Look up technical information from the course modules and lesson markdown files.
    """
    global search_call_count
    search_call_count += 1
    print(f"⚙️ Tool Call #{search_call_count}: Searching for query -> '{query}'")
    
    # Query your 295 chunks index
    results = chunk_index.search(query=query, num_results=3)
    return results

In [11]:
agent_instructions = (
    "You're a course teaching assistant. Answer the student's question using the "
    "search tool. Make multiple searches with different keywords before answering."
)

In [12]:
chat = client.chats.create(
    model="gemini-2.5-flash",
    config=dict(
        system_instruction=agent_instructions,
        tools=[search]
    )
)

# 3. Ask the target question
question = "How does the agentic loop work, and how is it different from plain RAG?"
print(f"👤 Student: {question}\n")

response = chat.send_message(question)

👤 Student: How does the agentic loop work, and how is it different from plain RAG?

⚙️ Tool Call #1: Searching for query -> 'agentic loop'
⚙️ Tool Call #2: Searching for query -> 'plain RAG'


In [13]:
print(f"\n✨ Final Answer:\n{response.text}\n")
print(f"📊 Total times 'search' tool was executed: {search_call_count}")


✨ Final Answer:
The agentic loop is a dynamic and iterative process where an AI agent (powered by a Large Language Model) continuously decides on actions, executes tools, and refines its approach based on the outcomes, until it reaches a final answer. This loop typically involves:

1.  **Calling the LLM:** The LLM receives a prompt and potentially previous messages.
2.  **Executing Tool Calls:** If the LLM determines that external tools (like a search engine or a code interpreter) are needed, it generates calls to these tools.
3.  **Sending Results Back:** The results from the tool executions are fed back to the LLM.
4.  **Iteration and Refinement:** The LLM processes the new information and decides whether to make more tool calls, refine its understanding, or generate a final answer. This process repeats until a satisfactory final answer is produced.

This approach is particularly useful when the exact sequence of steps to solve a problem is not known in advance, or when decisions ne